# Gnani public-dataset speech evaluation

This single notebook runs both ASR and TTS parts from released evidence without API keys. ASR uses original public dataset transcripts; TTS quality uses explicit human marks on selected scored samples. See README.md and METHODS.md for scope, source IDs, normalization and limitations.

In [1]:
from pathlib import Path
import csv, json, subprocess, sys
PROJECT = Path.cwd()
assert (PROJECT / "run_evaluation.py").is_file(), "Start Jupyter in the extracted project directory."
subprocess.run([sys.executable, "run_evaluation.py", "--mode", "replay", "--part", "all", "--reports"], check=True)
S = json.loads((PROJECT / "results/submission/summary.json").read_text(encoding="utf8"))
print({"Python": sys.version.split()[0], "profile": S["profile"], "new_api_calls": S["new_api_calls"]})

{'Python': '3.14.4', 'profile': 'public_submission', 'new_api_calls': 0}


## ASR: custom corpus WER/CER

Baseline contains 221 speech clips plus five no-speech controls. The 120 follow-up and 24 extension clips are separate, non-paired cohorts. Source utterances do not overlap; some speakers recur. Empty controls have undefined WER/CER and separate false-transcription outcomes.

In [2]:
import pandas as pd
asr = pd.DataFrame([{ "cohort": cohort, "speech_clips": r["speech_clips"], "no_speech": r["no_speech_controls"], "edits": r["word_edits"], "reference_words": r["reference_words"], "WER": r["wer"], "CER": r["cer"] } for cohort, r in S["asr"].items()])
display(asr)
assert (S["asr"]["baseline"]["word_edits"], S["asr"]["baseline"]["reference_words"]) == (309, 1630)
assert S["asr"]["baseline"]["no_speech_false_transcriptions"] == 0

,cohort,speech_clips,no_speech,edits,reference_words,WER,CER
0,baseline,221,5,309,1630,0.189571,0.108805
1,followup,120,0,186,881,0.211124,0.102546
2,public24,24,0,48,249,0.192771,0.132475


## TTS: quality on selected scored samples

The 164 generated clips are not all perceptually scored. Headline correctness uses 62 baseline and 48 follow-up human-marked targets. Ambiguity fails strict correctness but does not establish an omission. Naturalness/prosody are exploratory ratings on four distinct clips by one listener; they are not population MOS.

In [3]:
display(pd.DataFrame(S["tts_human"]))
display(pd.read_csv("results/submission/tts_human_quality_ratings.csv"))
print({"naturalness": S["naturalness"]["naturalness_mean"], "prosody": S["naturalness"]["prosody_mean"], "unique_clips": S["naturalness"]["unique_clips"], "listeners": S["naturalness"]["listeners"]})
display(pd.DataFrame([S["tts_audio_quality"]]))

,cohort,human_scored_prompts,human_scored_targets,correct_targets,wrong_targets,ambiguous_targets,strict_accuracy,critical_targets,critical_correct
0,baseline,38,62,57,4,1,0.919355,58,53
1,next,28,48,35,12,1,0.729167,48,35


,case_id,listener_id,rated_at_utc,naturalness_1to5,prosody_1to5,repeated_target_rows,source,blind_before_reveal,listener_language_qualification
0,T36,R1,2026-11-13T17:31:05.675Z,4,5,2,results/listening_ratings.csv,not_recorded,not_recorded
1,T43,R1,2026-11-13T17:31:24.736Z,4,5,1,results/listening_ratings.csv,false,not_recorded
2,T49,R1,2026-11-13T17:32:04.314Z,5,5,1,results/listening_ratings.csv,false,not_recorded
3,T70,R1,2026-11-13T17:30:44.495Z,5,5,3,results/listening_ratings.csv,not_recorded,not_recorded


{'naturalness': 4.5, 'prosody': 5, 'unique_clips': 4, 'listeners': 1}


,clips,decoded_nonempty,clips_with_near_full_scale_samples,clips_with_placeholder_header_lengths,clips_with_internal_low_energy_over_500ms,max_internal_low_energy_s,note
0,164,164,4,164,22,1.16,"PCM integrity/level/pause diagnostics, not per..."


## Playback and timing verification

Saved PCM and arrival timestamps are reconstructed independently. Twelve tested v3.0 streams underrun; two v2.5 controls do not. This configuration-specific client simulation does not establish live-handset reliability or model superiority. Buffer sensitivity uses 0/50/100/200 ms startup prebuffer and 1/10/20 ms reporting tolerances. Future-aware delayed replay is diagnostic, not an online buffering policy.

In [4]:
display(pd.read_csv("results/playback_validation/buffer_sensitivity_summary.csv"))
print({"ASR median ms": S["latency"]["asr_rest_latency_ms_median"], "TTS median ms": S["latency"]["tts_rest_latency_ms_median"], "TTS RTF": S["latency"]["tts_rest_rtf_median"]})
from IPython.display import HTML, display
display(HTML('<a href="results/playback_validation/index.html">Open 14-stream playback evidence</a>'))

,model,initial_prebuffer_ms,tolerance_ms,streams,streams_with_underflows,starvation_ms_min,starvation_ms_max
0,timbre-v3.0,0,1,12,12,844.833333,3910.916667
1,timbre-v2.5,0,1,2,0,0.000000,0.000000
2,timbre-v3.0,0,10,12,12,844.833333,3910.916667
3,timbre-v2.5,0,10,2,0,0.000000,0.000000
4,timbre-v3.0,0,20,12,12,844.833333,3910.916667
5,timbre-v2.5,0,20,2,0,0.000000,0.000000
6,timbre-v3.0,50,1,12,12,794.833333,3860.916667
7,timbre-v2.5,50,1,2,0,0.000000,0.000000
8,timbre-v3.0,50,10,12,12,794.833333,3860.916667
9,timbre-v2.5,50,10,2,0,0.000000,0.000000


{'ASR median ms': 192.1, 'TTS median ms': 1492.25, 'TTS RTF': 0.3722769982993197}


## Fresh Timbre 3.0 / 2.5 follow-up
73 cases / 74 HTTP attempts: 66 valid new clips, six empty HTTP-200 WAVs and one expected tag-only rejection. This cohort and provisional audio-model ratings are separate from the original human scores. The model missed a planted gap; chunk/PCM measurements establish playback timing. See TIMBRE_FOLLOWUP.md.

In [5]:
F = S["timbre_followup"]
assert F["new_api_calls_in_replay"] == 0
print({k: F[k] for k in ["status", "attempted_cases", "http_requests", "valid_audio_clips", "blind_judgments", "paired_judgments", "judge_gap_controls_all_passed"]})
print("REST full-body medians (seconds):", [(r["model"], r["length"], round(r["median_ms"]/1000, 3)) for r in F["rest_latency"]])
print("Fresh stream underruns:", [(r["case_id"], r["underruns"]) for r in F["fresh_streams"]])
display(HTML('<a href="results/timbre_followup/index.html">Open fresh clips and cue assessments</a>'))

{'status': 'completed', 'attempted_cases': 73, 'http_requests': 74, 'valid_audio_clips': 66, 'blind_judgments': 66, 'paired_judgments': 29, 'judge_gap_controls_all_passed': False}
REST full-body medians (seconds): [('timbre-v3.0', 'short', 5.043), ('timbre-v3.0', 'medium', 7.811), ('timbre-v3.0', 'long', 17.2), ('timbre-v2.5', 'short', 0.542), ('timbre-v2.5', 'medium', 1.074), ('timbre-v2.5', 'long', 2.75)]
Fresh stream underruns: [('NF-S-3-short', 2), ('NF-S-25-short', 0), ('NF-S-3-medium', 1), ('NF-S-25-medium', 0), ('NF-S-3-long', 3), ('NF-S-25-long', 0)]


## Report 4: controlled and held-out extension
Five-repeat controlled REST, warm short-prompt WS, disjoint HTTP audit, fixed-buffer sensitivity and an independent frozen-vocabulary ASR check. New TTS human-rating sheets remain blank. Evidence cohorts stay separate.

In [6]:
R4 = S["report4"]
H4 = S["report4_heldout"]
assert R4["new_api_calls_in_replay"] == H4["new_api_calls_in_replay"] == 0
assert all(r["successful_trials"] == 5 for r in R4["controlled_rest"])
display(pd.DataFrame(R4["controlled_rest"]))
display(pd.DataFrame(R4["http_cohorts"]).T)
print("Held-out ASR:", H4["asr"], H4["paired_outcomes"])
print("New human ratings:", H4["new_human_rating_rows"], "Scored held-out rewrite pairs:", H4["rewrite_pairs_scored"])
display(HTML('<a href="results/report4/index.html">Controlled timing and buffering</a> · <a href="results/report4_quality/listening.html">Blinded listening study</a>'))

,model,voice,length,input_chars,input_words,successful_trials,attempts,case_trials,attempt_success_rate,eventual_case_success_rate,request_total_ms_median,request_total_ms_min,request_total_ms_max,duration_s_median,duration_s_min,duration_s_max,rtf_client_median,rtf_client_min,rtf_client_max
0,timbre-v3.0,Tarika,short,61,10,5,5,5,1.0,1.0,3436.5,1004.6,5423.7,3.52,3.36,3.60,0.976278,0.292035,1.506583
1,timbre-v3.0,Tarika,medium,396,75,5,5,5,1.0,1.0,6504.2,3003.9,7599.4,22.72,14.80,23.44,0.324206,0.132214,0.439473
2,timbre-v3.0,Tarika,long,930,160,5,5,5,1.0,1.0,11747.1,9000.3,16007.3,52.88,40.32,54.64,0.222146,0.170202,0.374680
3,timbre-v2.5,Kaveri,short,61,10,5,5,5,1.0,1.0,766.6,730.6,1164.0,3.04,2.88,3.20,0.253681,0.229219,0.382895
4,timbre-v2.5,Kaveri,medium,396,75,5,5,5,1.0,1.0,1183.2,1164.1,1354.5,17.76,17.60,18.40,0.067227,0.064152,0.076267
5,timbre-v2.5,Kaveri,long,930,160,5,5,5,1.0,1.0,2563.6,2474.3,2944.3,42.56,41.92,43.36,0.059124,0.058209,0.069180


,cases,http_attempts,valid_pcm_attempts,disjoint_attempt_outcomes,http_status_counts,eventual_success_cases,attempt_success_rate,eventual_case_success_rate,positive_case_success_rate,expected_rejections,retry_attempts,explicit_recheck_cases,final_case_outcomes
timbre_followup,73,74,66,"{'valid_pcm': 66, 'empty_pcm': 6, 'http_error'...","{'200': 72, '500': 1, '400': 1}",66,0.891892,0.90411,0.916667,1,1,8,"{'success': 65, 'persistent_empty': 6, 'recove..."
report4_controlled,32,32,32,{'valid_pcm': 32},{'200': 32},32,1.0,1.0,1.0,0,0,0,{'success': 32}
report4_quality,54,54,53,"{'valid_pcm': 53, 'empty_pcm': 1}",{'200': 54},53,0.981481,0.981481,0.981481,0,0,0,"{'success': 53, 'persistent_empty': 1}"


Held-out ASR: {'baseline': {'speech_clips': 15, 'reference_words': 150, 'word_edits': 29, 'wer': 0.19333333333333333, 'cer': 0.15827338129496402, 'cer_no_spaces': 0.15178571428571427, 'target_terms_correct': 14, 'target_terms_total': 15, 'target_set_exact_clips': 14, 'critical_lexical_slots_correct_clips': 14, 'hinted_vocabulary_insertion_flags': 0}, 'hinted': {'speech_clips': 15, 'reference_words': 150, 'word_edits': 28, 'wer': 0.18666666666666668, 'cer': 0.15251798561151078, 'cer_no_spaces': 0.14642857142857144, 'target_terms_correct': 15, 'target_terms_total': 15, 'target_set_exact_clips': 15, 'critical_lexical_slots_correct_clips': 15, 'hinted_vocabulary_insertion_flags': 0}} {'unchanged': 14, 'improved': 1}
New human ratings: 0 Scored held-out rewrite pairs: 0


The report remains a guarded non-transactional support recommendation. Confirm critical values/actions. The small held-out ASR keyterm check is complete and reported separately. Independent listening, scored held-out TTS rewrites, generalized mitigation benefits, realistic online jitter-buffer/interruption behavior, handset tests and extended load remain future validation.